In [ ]:
import lmdb
import pickle
import argparse
import random
from pathlib import Path
from loguru import logger
import json
from utils import rotation_conversions as rc
import torch


In [ ]:
source_path = "path"
file_id = "id"

In [ ]:
src_env = lmdb.open(str(source_path), readonly=True, lock=False)
lmdb_list = []
with src_env.begin() as src_txn:
    # Get the unique fileid for each file and group the entries
    file_groups = {}
    cursor = src_txn.cursor()
    for key, value in cursor:
        idx = int(key.decode('ascii'))
        entry = pickle.loads(value)
        f_id = entry["seamless_id"]
        
        if f_id not in file_groups:
            file_groups[f_id] = []
        file_groups[f_id].append(idx)
    
    indices = file_groups[file_id]

    # Sort indices to ensure frames appear in order
    indices.sort() 
    for idx in indices:
        data = src_txn.get(f"{idx:08d}".encode("ascii"))
        lmdb_list.append(data)
